# Simple Chatbot Graph with LangGraph
A minimal chatbot implementation with Start -> Chat_Node -> End flow

## 1. Dependencies

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage

print("✓ Dependencies imported successfully")

## 2. Load Environment Variables

In [ ]:
# Load environment variables from parent directory
ENV_PATH = Path("/home/program/Documents/Tutor/Campus/5_LangSmith/.env")
load_dotenv(ENV_PATH)

api_key = os.getenv("OPENROUTER_API_KEY")

if api_key:
    print(f"✓ API Key loaded: {api_key[:20]}...")
else:
    print("✗ API Key not found")

## 3. Initialize LLM

In [ ]:
# Initialize the LLM with OpenRouter
llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
    model="nvidia/nemotron-3.5-lightning:free",
)

print("✓ LLM initialized with OpenRouter")
print(f"  Model: nvidia/nemotron-3.5-lightning:free")

## 4. Define Chat State

In [ ]:
# Define the state structure for the chatbot
class ChatState(TypedDict):
    """State structure for the chat graph"""
    message: Annotated[list[BaseMessage], add_messages]

print("✓ ChatState defined")

## 5. Define Chat Node

In [ ]:
def ChatNode(state: ChatState):
    """
    Chat node that processes user messages and returns LLM response.
    
    Args:
        state: ChatState containing list of messages
    
    Returns:
        dict with updated message list including LLM response
    """
    user_message = state['message']
    llm_response = llm.invoke(user_message)
    return {'message': [llm_response]}

print("✓ ChatNode function defined")

## 6. Build the Graph

In [ ]:
# Create the state graph
graph = StateGraph(ChatState)

# Add nodes
graph.add_node('ChatNode', ChatNode)

# Add edges: START -> ChatNode -> END
graph.add_edge(START, 'ChatNode')
graph.add_edge('ChatNode', END)

print("✓ Graph structure created: START -> ChatNode -> END")

## 7. Compile the Graph with Memory Checkpointer

In [ ]:
# Create a memory checkpointer for conversation history
checkpointer = InMemorySaver()

# Compile the graph
chatbot = graph.compile(checkpointer=checkpointer)

print("✓ Graph compiled with InMemorySaver checkpointer")

## 8. Test the Chatbot

In [ ]:
# Test the chatbot
test_message = "Hello! What is LangGraph?"
thread_id = "test-thread-1"

config = {
    'configurable': {
        'thread_id': thread_id
    }
}

print(f"\n📝 User Message: {test_message}")
print("\n⏳ Waiting for response...\n")

# Stream the response
response = None
for chunk in chatbot.stream(
    {"message": [HumanMessage(content=test_message)]},
    config=config,
    stream_mode='messages'
):
    message, metadata = chunk
    if hasattr(message, 'content') and message.content:
        print(f"🤖 Assistant: {message.content}")
        response = message.content

## 9. Continue Conversation (Multi-turn)

In [ ]:
# Continue the conversation with the same thread_id to maintain history
follow_up_message = "Can you give me a practical example?"

print(f"\n📝 User Message: {follow_up_message}")
print("\n⏳ Waiting for response...\n")

for chunk in chatbot.stream(
    {"message": [HumanMessage(content=follow_up_message)]},
    config=config,
    stream_mode='messages'
):
    message, metadata = chunk
    if hasattr(message, 'content') and message.content:
        print(f"🤖 Assistant: {message.content}")

## 10. Graph Visualization

In [ ]:
# Visualize the graph structure
try:
    from IPython.display import Image, display
    # Generate graph image
    graph_image = chatbot.get_graph().draw_mermaid_png()
    display(Image(graph_image))
    print("✓ Graph visualization displayed")
except Exception as e:
    print(f"Note: Visualization requires graphviz. Error: {e}")
    print("\nGraph Structure:")
    print("  START")
    print("    ↓")
    print("  ChatNode (processes messages with LLM)")
    print("    ↓")
    print("  END")